# Stage 5: Model Uncertainty with Deep Ensembles

This stage extends the confidence analysis from Stage 4.

In Stage 4, we studied the confidence produced by a single trained model and
examined how often that model was confidently incorrect on the external RSNA
evaluation set.

Here, we take the next step by training several independently initialised
copies of the same SmallCNN model.

The models will use the same training data, architecture, preprocessing and
training procedure, but different random seeds. Each model will then make
predictions on the same external RSNA studies.

The main idea is simple:

- If the models produce similar predictions, there is relatively little
  disagreement between them.
- If the models produce very different predictions, the disagreement can be
  used as a signal of model uncertainty.

The main research question for this stage is:

> Does model disagreement provide a useful signal for identifying uncertain
> or incorrect predictions under external distribution shift?

This is different from the confidence analysis in Stage 4. A single model
can be highly confident, while an ensemble of models may disagree strongly
about the same case.

This experiment therefore provides a first practical analysis of model
uncertainty using deep ensembles.

The ensemble is intentionally kept small so that the experiment remains
practical on the available CPU environment.

In [41]:
# NumPy is used for storing and analysing the predictions
# produced by the different ensemble members.
import numpy as np

# Pandas will help us organise the ensemble results into
# readable tables later in the notebook.
import pandas as pd

# Matplotlib will be used for the uncertainty visualisations.
import matplotlib.pyplot as plt

# PyTorch is used to recreate and train the SmallCNN model.
import torch
import torch.nn as nn

# DataLoader allows us to process the training data in batches.
from torch.utils.data import DataLoader

# torchvision provides the same image transformation used
# in the original Stage 1 experiment.
from torchvision import transforms

# PneumoniaMNIST is the dataset used to train the original model.
from medmnist import PneumoniaMNIST

# Scikit-learn provides the evaluation metrics used later
# when comparing ensemble predictions with the ground truth.
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    log_loss
)

print("Stage 5 imports loaded successfully.")

Stage 5 imports loaded successfully.


In [42]:
# We will use several different seeds for the ensemble.
#
# Each seed creates a different model initialisation and training
# sequence while keeping the rest of the experiment unchanged.

ENSEMBLE_SEEDS = [
    42,
    43,
    44,
    45,
    46
]


# Use the same device-selection approach as Stage 1.
#
# In the previous experiment this environment used the CPU,
# so the ensemble will normally train on the CPU as well.

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("Ensemble seeds:", ENSEMBLE_SEEDS)
print("Number of ensemble members:", len(ENSEMBLE_SEEDS))
print("Using device:", device)

Ensemble seeds: [42, 43, 44, 45, 46]
Number of ensemble members: 5
Using device: cpu


In [43]:
# Keep the project location consistent with the previous stages.
#
# We will save the ensemble model checkpoints and prediction
# results inside the same research project.

PROJECT_ROOT = (
    r"D:\0001_Kings_AI_Project"
    r"\trustworthy-ai-medical-imaging"
)


# Store ensemble checkpoints separately from the existing
# Stage 2 prediction file.
ensemble_checkpoint_dir = (
    PROJECT_ROOT
    + r"\research_data\stage5_ensemble_checkpoints"
)


# This will later contain the predictions from all ensemble members.
ensemble_prediction_path = (
    PROJECT_ROOT
    + r"\research_data\stage5_ensemble_predictions.npz"
)


print("Project root:")
print(PROJECT_ROOT)

print()
print("Ensemble checkpoint directory:")
print(ensemble_checkpoint_dir)

print()
print("Ensemble prediction file:")
print(ensemble_prediction_path)

Project root:
D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging

Ensemble checkpoint directory:
D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging\research_data\stage5_ensemble_checkpoints

Ensemble prediction file:
D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging\research_data\stage5_ensemble_predictions.npz


In [44]:
# Use the same image transformation as Stage 1.
#
# Keeping the preprocessing identical is important because the
# ensemble should differ only through its random initialisation,
# not because we changed how the X-rays are represented.

transform = transforms.ToTensor()


# Load the PneumoniaMNIST training data.
#
# These are the images from which each ensemble member will learn.
# Every model will see the same training dataset.

train_ds = PneumoniaMNIST(
    split="train",
    download=True,
    transform=transform,
    size=64
)


# Load the validation data.
#
# We keep this separate from the training data so that we can
# monitor the models during training without using the external
# RSNA dataset for model fitting.

val_ds = PneumoniaMNIST(
    split="val",
    download=True,
    transform=transform,
    size=64
)


# Use the same batch size as the original Stage 1 experiment.
#
# Keeping this unchanged makes the ensemble training setup
# directly comparable with the original model.

BATCH_SIZE = 64


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)


val_loader = DataLoader(
    val_ds,
    batch_size=128,
    shuffle=False
)


print("Training images:", len(train_ds))
print("Validation images:", len(val_ds))

print()
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

print()
print("Image size:", train_ds[0][0].shape)
print("Task:", train_ds.info["task"])
print("Labels:", train_ds.info["label"])

Using downloaded and verified file: C:\Users\PRASAD\.medmnist\pneumoniamnist_64.npz
Using downloaded and verified file: C:\Users\PRASAD\.medmnist\pneumoniamnist_64.npz
Training images: 4708
Validation images: 524

Training batches: 74
Validation batches: 5

Image size: torch.Size([1, 64, 64])
Task: binary-class
Labels: {'0': 'normal', '1': 'pneumonia'}


In [45]:
# Recreate the exact SmallCNN architecture used in Stage 1.
#
# The ensemble should use the same architecture as the original
# baseline model. The only intentional difference between ensemble
# members will be their random initialisation.

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # The model receives a single-channel grayscale X-ray image.
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            # Reduce the spatial dimensions to one value per feature.
            nn.AdaptiveAvgPool2d(1)
        )

        # One output value represents the pneumonia logit.
        self.classifier = nn.Linear(64, 1)

    def forward(self, x):
        x = self.features(x)

        # Flatten the feature representation while keeping
        # the batch dimension intact.
        x = torch.flatten(x, 1)

        # Remove the final dimension so the output is [batch_size].
        return self.classifier(x).squeeze(1)


# Create one model simply to verify that the architecture
# can be constructed correctly before training the ensemble.
model = SmallCNN().to(device)

print("SmallCNN model created successfully.")
print()
print(model)

SmallCNN model created successfully.

SmallCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): AdaptiveAvgPool2d(output_size=1)
  )
  (classifier): Linear(in_features=64, out_features=1, bias=True)
)


In [46]:
# Define the same optimisation setup used in Stage 1.
#
# Keeping the loss function, learning rate, and number of epochs
# unchanged makes the ensemble members directly comparable with
# the original Stage 1 model.

criterion = nn.BCEWithLogitsLoss()

LEARNING_RATE = 1e-3
NUM_EPOCHS = 5

print("Loss function:", criterion)
print("Learning rate:", LEARNING_RATE)
print("Number of epochs:", NUM_EPOCHS)

Loss function: BCEWithLogitsLoss()
Learning rate: 0.001
Number of epochs: 5


In [47]:
# Train five independently initialised models.
#
# The purpose of a deep ensemble is to see how much predictions
# change when the same learning problem is solved from different
# random starting points. Each member therefore uses the same
# training procedure but a different random seed.

import os
import random

# Create the checkpoint directory if it does not already exist.
os.makedirs(ensemble_checkpoint_dir, exist_ok=True)

ensemble_training_results = []


for seed in ENSEMBLE_SEEDS:

    print("=" * 70)
    print(f"Training ensemble member with seed {seed}")
    print("=" * 70)

    # Set all relevant random seeds so that each ensemble member
    # starts from a reproducible but different initialisation.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Create a fresh model for this ensemble member.
    model = SmallCNN().to(device)

    # Each member gets its own optimiser so that training starts
    # independently from the other ensemble members.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    epoch_results = []

    for epoch in range(NUM_EPOCHS):

        model.train()
        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)

            # Convert the dataset labels to floating-point values
            # because BCEWithLogitsLoss expects floating-point targets.
            labels = labels.float().to(device).view(-1)

            optimizer.zero_grad()

            logits = model(images)

            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(train_loader.dataset)


        # Evaluate the current model on the validation set.
        # The validation data is not used to update model weights.
        model.eval()

        validation_loss = 0.0

        with torch.no_grad():
            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.float().to(device).view(-1)

                logits = model(images)

                loss = criterion(logits, labels)

                validation_loss += loss.item() * images.size(0)

        validation_loss /= len(val_loader.dataset)

        epoch_results.append(
            {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "validation_loss": validation_loss
            }
        )

        print(
            f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Validation Loss: {validation_loss:.4f}"
        )


    # Save the trained weights so that every ensemble member can
    # be independently inspected or reused later without retraining.
    checkpoint_path = (
        ensemble_checkpoint_dir
        + f"\\model_seed_{seed}.pth"
    )

    torch.save(
        model.state_dict(),
        checkpoint_path
    )

    ensemble_training_results.append(
        {
            "seed": seed,
            "checkpoint": checkpoint_path,
            "final_train_loss": epoch_results[-1]["train_loss"],
            "final_validation_loss": epoch_results[-1]["validation_loss"]
        }
    )

    print(f"Checkpoint saved: {checkpoint_path}")
    print()


print("=" * 70)
print("All ensemble members trained successfully.")
print("=" * 70)

training_summary = pd.DataFrame(ensemble_training_results)
print(training_summary)

Training ensemble member with seed 42
Epoch 1/5 | Train Loss: 0.5903 | Validation Loss: 0.5673
Epoch 2/5 | Train Loss: 0.5669 | Validation Loss: 0.5596
Epoch 3/5 | Train Loss: 0.5509 | Validation Loss: 0.5460
Epoch 4/5 | Train Loss: 0.5207 | Validation Loss: 0.4934
Epoch 5/5 | Train Loss: 0.4845 | Validation Loss: 0.4497
Checkpoint saved: D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging\research_data\stage5_ensemble_checkpoints\model_seed_42.pth

Training ensemble member with seed 43
Epoch 1/5 | Train Loss: 0.5796 | Validation Loss: 0.5670
Epoch 2/5 | Train Loss: 0.5502 | Validation Loss: 0.5271
Epoch 3/5 | Train Loss: 0.4287 | Validation Loss: 0.3579
Epoch 4/5 | Train Loss: 0.3342 | Validation Loss: 0.2859
Epoch 5/5 | Train Loss: 0.2856 | Validation Loss: 0.2775
Checkpoint saved: D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging\research_data\stage5_ensemble_checkpoints\model_seed_43.pth

Training ensemble member with seed 44
Epoch 1/5 | Train Loss: 0.5855 | Validation Lo

In [48]:
# Load the five independently trained ensemble members and
# generate predictions on the same RSNA external dataset used
# throughout Stages 2–4.
#
# We deliberately reuse the saved Stage 2 predictions only for
# comparison later. The ensemble itself must produce its own
# predictions so that we can measure model-to-model disagreement.

# Load the RSNA dataset information from the Stage 2 prediction file.
#
# The labels are retained only as ground truth for evaluation.
# The ensemble does not use them during prediction.

stage2_predictions = np.load(
    ensemble_prediction_path.replace(
        "stage5_ensemble_predictions.npz",
        "rsna_stage2_predictions.npz"
    )
)

all_labels = stage2_predictions["labels"]

print("RSNA labels loaded successfully.")
print("Number of RSNA studies:", len(all_labels))
print("Label values:", np.unique(all_labels))


# The actual RSNA DICOM inference dataset is reconstructed using
# the same preprocessing used in Stage 2.
#
# The next cells will load the DICOM images and match them to the
# saved study-level labels. We first verify that the expected
# external sample count is available.

print()
print("Expected ensemble prediction count:", len(all_labels))

RSNA labels loaded successfully.
Number of RSNA studies: 25684
Label values: [0 1]

Expected ensemble prediction count: 25684


In [49]:
# Re-establish the RSNA dataset paths.
#
# These are the same locations used by the validated Stage 2
# external evaluation. Keeping the paths explicit here makes this
# notebook easier to reproduce if the kernel is restarted.

from pathlib import Path

PROJECT_ROOT = Path(
    r"D:\0001_Kings_AI_Project"
    r"\trustworthy-ai-medical-imaging"
)

RSNA_ROOT = (
    PROJECT_ROOT.parent
    / "datasets"
    / "rsna_pneumonia"
)

dicom_root = (
    RSNA_ROOT
    / "raw"
    / "images"
)

annotation_path = (
    RSNA_ROOT
    / "annotations"
    / "pneumonia-challenge-annotations-original_2018.json"
)

print("RSNA root:")
print(RSNA_ROOT)

print()
print("DICOM root:")
print(dicom_root)

print()
print("Annotation file:")
print(annotation_path)

print()
print("DICOM directory exists:", dicom_root.exists())
print("Annotation file exists:", annotation_path.exists())

RSNA root:
D:\0001_Kings_AI_Project\datasets\rsna_pneumonia

DICOM root:
D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\raw\images

Annotation file:
D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\annotations\pneumonia-challenge-annotations-original_2018.json

DICOM directory exists: True
Annotation file exists: True


In [50]:
# Inspect the actual contents of the RSNA image directory.
#
# The completed Stage 2 notebook found 25,684 DICOM files here,
# so before changing anything else we want to see how the files
# are currently organised on disk.

print("RSNA image directory:")
print(dicom_root)

print()
print("Directory exists:", dicom_root.exists())

if dicom_root.exists():

    # Show the first few items directly inside the directory.
    direct_items = list(dicom_root.iterdir())

    print("Items directly inside directory:", len(direct_items))
    print()

    for item in direct_items[:10]:
        print(
            " -",
            item.name,
            "|",
            "directory" if item.is_dir() else "file"
        )

    print()

    # Search recursively for files regardless of extension.
    # This helps us determine whether the DICOM files have an
    # unexpected extension or are stored under another level.
    all_files = [
        path
        for path in dicom_root.rglob("*")
        if path.is_file()
    ]

    print("Total files found recursively:", len(all_files))

    print()
    print("First 10 recursive files:")

    for path in all_files[:10]:
        print(" -", path)

else:
    print("The RSNA image directory does not exist.")

RSNA image directory:
D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\raw\images

Directory exists: True
Items directly inside directory: 1

 - mdai_public_project_LxR6zdR2_images_2018-08-20-184248 | directory

Total files found recursively: 25684

First 10 recursive files:
 - D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\raw\images\mdai_public_project_LxR6zdR2_images_2018-08-20-184248\1.2.276.0.7230010.3.1.2.8323329.10001.1517874346.163715\1.2.276.0.7230010.3.1.3.8323329.10001.1517874346.163714\1.2.276.0.7230010.3.1.4.8323329.10001.1517874346.163716.dcm
 - D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\raw\images\mdai_public_project_LxR6zdR2_images_2018-08-20-184248\1.2.276.0.7230010.3.1.2.8323329.10002.1517874346.165591\1.2.276.0.7230010.3.1.3.8323329.10002.1517874346.165590\1.2.276.0.7230010.3.1.4.8323329.10002.1517874346.165592.dcm
 - D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\raw\images\mdai_public_project_LxR6zdR2_images_2018-08-20-184248\1.2.276.0.7230010.3.1.2.83233

In [51]:
# Discover every RSNA DICOM file recursively.
#
# The RSNA archive contains several nested directory levels,
# so a recursive search is required to recover all 25,684 images.

dicom_files = sorted(
    dicom_root.rglob("*.dcm")
)

print("DICOM files found:", len(dicom_files))

print()
print("Expected DICOM files:", 25684)

print()
print("File count matches Stage 2:", len(dicom_files) == 25684)

print()
print("First DICOM file:")
print(dicom_files[0] if dicom_files else "No DICOM files found")

print()
print("Last DICOM file:")
print(dicom_files[-1] if dicom_files else "No DICOM files found")

DICOM files found: 25684

Expected DICOM files: 25684

File count matches Stage 2: True

First DICOM file:
D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\raw\images\mdai_public_project_LxR6zdR2_images_2018-08-20-184248\1.2.276.0.7230010.3.1.2.8323329.10001.1517874346.163715\1.2.276.0.7230010.3.1.3.8323329.10001.1517874346.163714\1.2.276.0.7230010.3.1.4.8323329.10001.1517874346.163716.dcm

Last DICOM file:
D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\raw\images\mdai_public_project_LxR6zdR2_images_2018-08-20-184248\1.2.276.0.7230010.3.1.2.8323329.9999.1517874346.142251\1.2.276.0.7230010.3.1.3.8323329.9999.1517874346.142250\1.2.276.0.7230010.3.1.4.8323329.9999.1517874346.142252.dcm


In [52]:
# Load the libraries needed for the RSNA DICOM work.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom

from pathlib import Path
from PIL import Image

print("Required libraries loaded successfully.")

Required libraries loaded successfully.


In [53]:
# Inspect a small sample of DICOM metadata.
#
# We only read metadata here, not the image pixels. This lets us
# confirm that each DICOM contains the identifiers needed to connect
# the image to its RSNA study-level ground-truth label.

for path in dicom_files[:5]:

    ds = pydicom.dcmread(
        path,
        stop_before_pixels=True
    )

    print("\nFile:", path.name)

    print(
        "StudyInstanceUID:",
        getattr(ds, "StudyInstanceUID", None)
    )

    print(
        "SeriesInstanceUID:",
        getattr(ds, "SeriesInstanceUID", None)
    )

    print(
        "SOPInstanceUID:",
        getattr(ds, "SOPInstanceUID", None)
    )

    print(
        "Rows:",
        getattr(ds, "Rows", None)
    )

    print(
        "Columns:",
        getattr(ds, "Columns", None)
    )

    print(
        "PhotometricInterpretation:",
        getattr(ds, "PhotometricInterpretation", None)
    )


File: 1.2.276.0.7230010.3.1.4.8323329.10001.1517874346.163716.dcm
StudyInstanceUID: 1.2.276.0.7230010.3.1.2.8323329.10001.1517874346.163715
SeriesInstanceUID: 1.2.276.0.7230010.3.1.3.8323329.10001.1517874346.163714
SOPInstanceUID: 1.2.276.0.7230010.3.1.4.8323329.10001.1517874346.163716
Rows: 1024
Columns: 1024
PhotometricInterpretation: MONOCHROME2

File: 1.2.276.0.7230010.3.1.4.8323329.10002.1517874346.165592.dcm
StudyInstanceUID: 1.2.276.0.7230010.3.1.2.8323329.10002.1517874346.165591
SeriesInstanceUID: 1.2.276.0.7230010.3.1.3.8323329.10002.1517874346.165590
SOPInstanceUID: 1.2.276.0.7230010.3.1.4.8323329.10002.1517874346.165592
Rows: 1024
Columns: 1024
PhotometricInterpretation: MONOCHROME2

File: 1.2.276.0.7230010.3.1.4.8323329.10004.1517874346.174866.dcm
StudyInstanceUID: 1.2.276.0.7230010.3.1.2.8323329.10004.1517874346.174865
SeriesInstanceUID: 1.2.276.0.7230010.3.1.3.8323329.10004.1517874346.174864
SOPInstanceUID: 1.2.276.0.7230010.3.1.4.8323329.10004.1517874346.174866
Rows: 10

In [54]:
# Load the RSNA annotation JSON.
#
# This creates the `annotations` variable used by the following
# structure-inspection cell.

import json

with open(
    annotation_path,
    "r",
    encoding="utf-8"
) as f:
    annotations = json.load(f)

print("Annotation file loaded successfully.")
print("Top-level type:", type(annotations).__name__)

Annotation file loaded successfully.
Top-level type: dict


In [55]:
# The RSNA annotation JSON is organised as a dictionary.
# The actual annotation records are stored inside the first dataset.
#
# We first inspect the structure before building the label lookup.
# This avoids accidentally treating the eight top-level entries
# as the eight annotation records.

print("Top-level type:", type(annotations).__name__)

print()
print("Top-level keys:")
print(list(annotations.keys()))

print()
print("Number of datasets:", len(annotations["datasets"]))

print(
    "Number of studies:",
    len(annotations["datasets"][0]["studies"])
)

print(
    "Number of annotations:",
    len(annotations["datasets"][0]["annotations"])
)

Top-level type: dict

Top-level keys:
['id', 'createdAt', 'updatedAt', 'name', 'description', 'isPrivate', 'labelGroups', 'datasets']

Number of datasets: 1
Number of studies: 25684
Number of annotations: 28989


In [56]:
# Build a lookup from the RSNA label ID to its human-readable name.
#
# We use the label definitions provided inside the annotation file
# so that the study-level labels are constructed from the dataset's
# actual definitions rather than from manually assumed label IDs.

label_map = {}

for label in annotations["labelGroups"][0]["labels"]:
    label_id = label["id"]
    label_name = label["name"]

    label_map[label_id] = label_name

print("RSNA label definitions loaded successfully.")
print()

for label_id, label_name in label_map.items():
    print(f"Label ID: {label_id}")
    print(f"Label name: {label_name}")
    print()

RSNA label definitions loaded successfully.

Label ID: L_ylR0L8
Label name: No Lung Opacity

Label ID: L_DlqEAl
Label name: Lung Opacity



In [57]:
from collections import Counter

# Extract the annotation records from the first RSNA dataset.
annotations_list = annotations["datasets"][0]["annotations"]

# Count how many annotation records belong to each label.
annotation_label_counts = Counter(
    label_map[annotation["labelId"]]
    for annotation in annotations_list
)

print("Annotation counts by label:")
for label_name, count in annotation_label_counts.items():
    print(f"  {label_name}: {count}")

print()
print("Total annotation records:", len(annotations_list))

Annotation counts by label:
  No Lung Opacity: 20025
  Lung Opacity: 8964

Total annotation records: 28989


In [58]:
from collections import defaultdict

# Group all annotation records belonging to the same study.
#
# A study can contain multiple annotation records, so we first
# collect them together using StudyInstanceUID as the linking key.

study_annotations = defaultdict(list)

for annotation in annotations_list:
    study_uid = annotation["StudyInstanceUID"]
    study_annotations[study_uid].append(annotation)

print("Annotations grouped by study successfully.")
print()
print("Studies with annotations:", len(study_annotations))

print()
print("Total annotation records recovered:")
print(sum(len(records) for records in study_annotations.values()))

Annotations grouped by study successfully.

Studies with annotations: 25684

Total annotation records recovered:
28989


In [59]:
# Check which combinations of labels occur within each study.
#
# This is an important validation step because a study can contain
# multiple annotation records. We want to know whether a study ever
# contains both "Lung Opacity" and "No Lung Opacity" annotations.

label_combinations = Counter()

for study_uid, study_annots in study_annotations.items():

    labels_in_study = sorted(
        {
            label_map[annotation["labelId"]]
            for annotation in study_annots
        }
    )

    combination = " + ".join(labels_in_study)
    label_combinations[combination] += 1

print("Label combinations by study:")
print()

for combination, count in label_combinations.most_common():
    print(f"  {combination}: {count}")

print()
print("Total studies represented:", sum(label_combinations.values()))

Label combinations by study:

  No Lung Opacity: 20025
  Lung Opacity: 5659

Total studies represented: 25684


In [60]:
# Create one binary ground-truth label for each RSNA study.
#
# The annotation validation above showed that each study contains
# exactly one of the two label types, so the mapping is unambiguous:
#
#   0 = No Lung Opacity
#   1 = Lung Opacity

study_rows = []

for study_uid, study_annots in study_annotations.items():

    label_names = {
        label_map[annotation["labelId"]]
        for annotation in study_annots
    }

    if "Lung Opacity" in label_names:
        label = 1
        label_name = "Lung Opacity"

    elif "No Lung Opacity" in label_names:
        label = 0
        label_name = "No Lung Opacity"

    else:
        # Unexpected labels are excluded rather than being assigned
        # a class that the dataset did not explicitly provide.
        continue

    study_rows.append({
        "StudyInstanceUID": study_uid,
        "label": label,
        "label_name": label_name,
        "annotation_count": len(study_annots)
    })

rsna_labels = pd.DataFrame(study_rows)

print("RSNA study-level dataset created.")
print()
print("Number of studies:", len(rsna_labels))

print()
print("Study-level label distribution:")
print(rsna_labels["label_name"].value_counts())

print()
print("Label values:", sorted(rsna_labels["label"].unique()))

RSNA study-level dataset created.

Number of studies: 25684

Study-level label distribution:
label_name
No Lung Opacity    20025
Lung Opacity        5659
Name: count, dtype: int64

Label values: [np.int64(0), np.int64(1)]


In [61]:
# Create a quick lookup from StudyInstanceUID to the validated
# study-level ground-truth label.
#
# This lets the ensemble inference loop attach the correct label
# to each DICOM image without repeatedly searching the DataFrame.

study_label_lookup = dict(
    zip(
        rsna_labels["StudyInstanceUID"],
        rsna_labels["label"]
    )
)

print("Study-level label lookup created.")
print("Studies in lookup:", len(study_label_lookup))

print()
print("Example StudyInstanceUID:")
example_uid = rsna_labels.iloc[0]["StudyInstanceUID"]
print(example_uid)

print()
print("Example label:")
print(study_label_lookup[example_uid])

Study-level label lookup created.
Studies in lookup: 25684

Example StudyInstanceUID:
1.2.276.0.7230010.3.1.2.8323329.5872.1517874318.729899

Example label:
0


In [62]:
# Verify that every RSNA DICOM image can be linked to one of the
# validated study-level labels.
#
# We only read DICOM metadata here, not pixel data. This keeps the
# validation step lightweight while checking the identifier linkage.

import pydicom

matched_dicom_count = 0
missing_label_count = 0
study_ids_seen = set()

for path in dicom_files:

    ds = pydicom.dcmread(
        path,
        stop_before_pixels=True
    )

    study_uid = getattr(
        ds,
        "StudyInstanceUID",
        None
    )

    if study_uid in study_label_lookup:
        matched_dicom_count += 1
        study_ids_seen.add(study_uid)
    else:
        missing_label_count += 1

print("DICOM-to-label linkage check complete.")
print()
print("DICOM files checked:", len(dicom_files))
print("DICOM files with matching labels:", matched_dicom_count)
print("DICOM files without matching labels:", missing_label_count)
print()
print("Unique study IDs found in DICOM files:", len(study_ids_seen))

DICOM-to-label linkage check complete.

DICOM files checked: 25684
DICOM files with matching labels: 25684
DICOM files without matching labels: 0

Unique study IDs found in DICOM files: 25684


In [63]:
# Use exactly the same preprocessing as the Stage 2 RSNA evaluation.
#
# Stage 2 read the DICOM pixel array directly, treated it as a
# grayscale image, resized it to 64 x 64, and converted it to a
# PyTorch tensor.
#
# We deliberately do NOT perform additional min-max normalization
# here because that would change the input distribution relative
# to the original external evaluation.

rsna_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])


def load_rsna_image(dicom_path):
    """
    Read one RSNA DICOM image and apply the same preprocessing
    used during the validated Stage 2 external evaluation.
    """

    ds = pydicom.dcmread(dicom_path)

    # Extract the original DICOM pixel array.
    pixel_array = ds.pixel_array

    # Treat the RSNA image as a single-channel grayscale image,
    # matching the preprocessing used in Stage 2.
    image = Image.fromarray(
        pixel_array,
        mode="L"
    )

    # Resize to the 64 x 64 input size expected by SmallCNN
    # and convert the image into a PyTorch tensor.
    tensor = rsna_transform(image)

    return tensor

In [64]:
# Recreate the list of RSNA DICOM files.
#
# The RSNA dataset contains the DICOM images inside nested folders,
# so we search recursively rather than looking only in the top folder.

from pathlib import Path

RSNA_IMAGE_ROOT = Path(
    r"D:\0001_Kings_AI_Project"
    r"\datasets\rsna_pneumonia"
    r"\raw\images"
)

rsna_dicom_paths = sorted(
    RSNA_IMAGE_ROOT.rglob("*.dcm")
)

print("RSNA DICOM discovery completed.")
print("Number of DICOM files:", len(rsna_dicom_paths))

print()
print("First DICOM file:")
print(rsna_dicom_paths[0])

RSNA DICOM discovery completed.
Number of DICOM files: 25684

First DICOM file:
D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\raw\images\mdai_public_project_LxR6zdR2_images_2018-08-20-184248\1.2.276.0.7230010.3.1.2.8323329.10001.1517874346.163715\1.2.276.0.7230010.3.1.3.8323329.10001.1517874346.163714\1.2.276.0.7230010.3.1.4.8323329.10001.1517874346.163716.dcm


In [65]:
# Test the corrected preprocessing on one RSNA DICOM image.
#
# We are checking that the image can be read successfully and that
# the resulting tensor has the same basic format expected by SmallCNN.

sample_dicom = rsna_dicom_paths[0]

sample_tensor = load_rsna_image(sample_dicom)

print("Sample DICOM:")
print(sample_dicom)

print()
print("Tensor shape:", sample_tensor.shape)
print("Tensor dtype:", sample_tensor.dtype)
print("Tensor minimum:", sample_tensor.min().item())
print("Tensor maximum:", sample_tensor.max().item())

Sample DICOM:
D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\raw\images\mdai_public_project_LxR6zdR2_images_2018-08-20-184248\1.2.276.0.7230010.3.1.2.8323329.10001.1517874346.163715\1.2.276.0.7230010.3.1.3.8323329.10001.1517874346.163714\1.2.276.0.7230010.3.1.4.8323329.10001.1517874346.163716.dcm

Tensor shape: torch.Size([1, 64, 64])
Tensor dtype: torch.float32
Tensor minimum: 0.0
Tensor maximum: 0.8117647171020508


In [66]:
# Load the five independently trained ensemble members.
#
# Each checkpoint uses the same SmallCNN architecture but was trained
# with a different random seed. The resulting differences between the
# models will later be used to measure model disagreement.

ensemble_models = []

for seed in ENSEMBLE_SEEDS:

    checkpoint_path = (
        ensemble_checkpoint_dir
        + f"\\model_seed_{seed}.pth"
    )

    model = SmallCNN().to(device)

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device
        )
    )

    model.eval()

    ensemble_models.append(model)

    print(f"Loaded ensemble member: seed {seed}")

print()
print("Total ensemble members loaded:", len(ensemble_models))

Loaded ensemble member: seed 42
Loaded ensemble member: seed 43
Loaded ensemble member: seed 44
Loaded ensemble member: seed 45
Loaded ensemble member: seed 46

Total ensemble members loaded: 5


C:\Users\PRASAD\AppData\Local\Temp\ipykernel_4068\2209865673.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


In [67]:
# Run the five ensemble members over the RSNA images.
#
# Each model produces one probability for every study. We keep the
# predictions from each model separately so that we can later
# calculate both the ensemble mean and the disagreement between models.

INFERENCE_BATCH_SIZE = 64

ensemble_probability_predictions = []

print("Starting ensemble inference...")
print()
print("Number of DICOM files:", len(dicom_files))
print("Number of ensemble members:", len(ensemble_models))
print("Inference batch size:", INFERENCE_BATCH_SIZE)
print()

for model_index, model in enumerate(ensemble_models, start=1):

    print(
        f"Running ensemble member "
        f"{model_index}/{len(ensemble_models)}..."
    )

    model_probabilities = []

    for start in range(
        0,
        len(dicom_files),
        INFERENCE_BATCH_SIZE
    ):

        batch_paths = dicom_files[
            start:start + INFERENCE_BATCH_SIZE
        ]

        batch_images = torch.stack(
            [
                load_rsna_image(path)
                for path in batch_paths
            ]
        ).to(device)

        with torch.no_grad():
            logits = model(batch_images)
            probabilities = torch.sigmoid(logits)

        model_probabilities.extend(
            probabilities.cpu().numpy()
        )

    model_probabilities = np.asarray(
        model_probabilities,
        dtype=np.float32
    )

    ensemble_probability_predictions.append(
        model_probabilities
    )

    print(
        f"Completed member {model_index}. "
        f"Predictions: {len(model_probabilities)}"
    )

print()
print("Ensemble inference completed.")

Starting ensemble inference...

Number of DICOM files: 25684
Number of ensemble members: 5
Inference batch size: 64

Running ensemble member 1/5...
Completed member 1. Predictions: 25684
Running ensemble member 2/5...
Completed member 2. Predictions: 25684
Running ensemble member 3/5...
Completed member 3. Predictions: 25684
Running ensemble member 4/5...
Completed member 4. Predictions: 25684
Running ensemble member 5/5...
Completed member 5. Predictions: 25684

Ensemble inference completed.


In [68]:
# Convert the five model prediction arrays into one prediction matrix.
#
# Each row represents one RSNA study.
# Each column represents one independently trained ensemble member.
#
# This structure lets us measure how much the models agree or disagree
# about each individual study.

ensemble_probabilities = np.vstack(
    ensemble_probability_predictions
).T

print("Ensemble prediction matrix created.")
print()
print("Matrix shape:", ensemble_probabilities.shape)

print()
print("Expected shape:")
print(
    (
        len(dicom_files),
        len(ensemble_models)
    )
)

# Calculate the mean probability across the five models.
ensemble_mean_probability = np.mean(
    ensemble_probabilities,
    axis=1
)

# Calculate the variance between model predictions.
# Higher variance means stronger disagreement between ensemble members.
ensemble_variance = np.var(
    ensemble_probabilities,
    axis=1
)

# Standard deviation is easier to interpret on the same probability scale.
ensemble_std = np.std(
    ensemble_probabilities,
    axis=1
)

# Convert the ensemble mean probability into a binary prediction.
ensemble_predictions = (
    ensemble_mean_probability >= 0.5
).astype(int)

print()
print("Ensemble mean probability range:")
print(
    f"{ensemble_mean_probability.min():.6f} "
    f"to "
    f"{ensemble_mean_probability.max():.6f}"
)

print()
print("Ensemble variance range:")
print(
    f"{ensemble_variance.min():.6f} "
    f"to "
    f"{ensemble_variance.max():.6f}"
)

print()
print("Ensemble standard deviation range:")
print(
    f"{ensemble_std.min():.6f} "
    f"to "
    f"{ensemble_std.max():.6f}"
)

print()
print("Predicted class counts:")
print(
    pd.Series(ensemble_predictions).value_counts().sort_index()
)

Ensemble prediction matrix created.

Matrix shape: (25684, 5)

Expected shape:
(25684, 5)

Ensemble mean probability range:
0.116930 to 0.986835

Ensemble variance range:
0.000174 to 0.073641

Ensemble standard deviation range:
0.013174 to 0.271369

Predicted class counts:
0     6621
1    19063
Name: count, dtype: int64


In [69]:
# Evaluate the ensemble predictions against the validated RSNA labels.
#
# This gives us the basic predictive performance of the ensemble
# before we analyse disagreement as an uncertainty signal.

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    log_loss
)

# Use the study-level labels created and validated earlier.
ensemble_labels = rsna_labels["label"].to_numpy()

ensemble_accuracy = accuracy_score(
    ensemble_labels,
    ensemble_predictions
)

ensemble_auroc = roc_auc_score(
    ensemble_labels,
    ensemble_mean_probability
)

ensemble_brier = brier_score_loss(
    ensemble_labels,
    ensemble_mean_probability
)

ensemble_logloss = log_loss(
    ensemble_labels,
    ensemble_mean_probability
)

print("Ensemble performance on RSNA:")
print()
print(f"Accuracy:  {ensemble_accuracy:.4f}")
print(f"AUROC:     {ensemble_auroc:.4f}")
print(f"Brier:     {ensemble_brier:.4f}")
print(f"Log Loss:  {ensemble_logloss:.4f}")

print()
print("Number of evaluated studies:", len(ensemble_labels))
print(
    "Prediction/label length match:",
    len(ensemble_predictions) == len(ensemble_labels)
)

Ensemble performance on RSNA:

Accuracy:  0.3611
AUROC:     0.4938
Brier:     0.4282
Log Loss:  1.2086

Number of evaluated studies: 25684
Prediction/label length match: True


In [72]:
# Compare the corrected seed-42 ensemble predictions with the
# original Stage 2 predictions.
#
# PROJECT_ROOT is a pathlib.Path object, so we build the file path
# using "/" rather than concatenating a Windows string manually.

stage2_prediction_path = (
    PROJECT_ROOT
    / "research_data"
    / "rsna_stage2_predictions.npz"
)

original_stage2 = np.load(stage2_prediction_path)

original_probabilities = original_stage2["probabilities"]

# The first row contains predictions from ensemble member seed 42.
seed42_probabilities = ensemble_probability_predictions[0]

print("Stage 2 prediction file:")
print(stage2_prediction_path)

print()
print("Number of Stage 2 predictions:", len(original_probabilities))
print("Number of seed-42 predictions:", len(seed42_probabilities))

print()
print(
    "Mean absolute difference:",
    np.mean(
        np.abs(
            original_probabilities
            - seed42_probabilities
        )
    )
)

print(
    "Maximum absolute difference:",
    np.max(
        np.abs(
            original_probabilities
            - seed42_probabilities
        )
    )
)

Stage 2 prediction file:
D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging\research_data\rsna_stage2_predictions.npz

Number of Stage 2 predictions: 25684
Number of seed-42 predictions: 25684

Mean absolute difference: 0.0
Maximum absolute difference: 0.0


In [73]:
# Evaluate each ensemble member separately on the RSNA external dataset.
#
# This helps us understand whether the poor ensemble performance comes
# from all individual models or from averaging models that behave
# differently under the external distribution shift.

member_results = []

for member_index, seed in enumerate(ENSEMBLE_SEEDS):

    member_probabilities = ensemble_probability_predictions[member_index]

    member_predictions = (
        member_probabilities >= 0.5
    ).astype(int)

    member_accuracy = accuracy_score(
        all_labels,
        member_predictions
    )

    member_auroc = roc_auc_score(
        all_labels,
        member_probabilities
    )

    member_brier = brier_score_loss(
        all_labels,
        member_probabilities
    )

    member_logloss = log_loss(
        all_labels,
        member_probabilities
    )

    member_results.append({
        "seed": seed,
        "accuracy": member_accuracy,
        "auroc": member_auroc,
        "brier": member_brier,
        "log_loss": member_logloss
    })

member_results_df = pd.DataFrame(member_results)

print("Individual ensemble-member performance on RSNA:")
print(
    member_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

Individual ensemble-member performance on RSNA:
 seed  accuracy  auroc  brier  log_loss
   42    0.3639 0.5640 0.3579    0.9652
   43    0.5014 0.6670 0.3768    1.2241
   44    0.3940 0.6454 0.4602    1.5487
   45    0.3844 0.6615 0.3628    0.9893
   46    0.4199 0.6413 0.4405    1.4412


In [74]:
# Calculate the ensemble's mean prediction and disagreement.
#
# The ensemble mean represents the average prediction across the
# five independently trained models.
#
# The variance and standard deviation measure how much the models
# disagree with one another for each RSNA study.
#
# Higher disagreement means the models are less consistent about
# the prediction for that particular image.

ensemble_mean_probability = np.mean(
    ensemble_probability_predictions,
    axis=0
)

ensemble_variance = np.var(
    ensemble_probability_predictions,
    axis=0
)

ensemble_standard_deviation = np.std(
    ensemble_probability_predictions,
    axis=0
)

# Convert the ensemble mean probability into the final binary
# prediction using the same 0.5 threshold used earlier.

ensemble_predictions = (
    ensemble_mean_probability >= 0.5
).astype(int)

# Check whether the arrays have the expected dimensions.

print("Ensemble uncertainty analysis created.")

print()
print("Mean probability shape:")
print(ensemble_mean_probability.shape)

print()
print("Variance range:")
print(
    f"{ensemble_variance.min():.6f} "
    f"to "
    f"{ensemble_variance.max():.6f}"
)

print()
print("Standard deviation range:")
print(
    f"{ensemble_standard_deviation.min():.6f} "
    f"to "
    f"{ensemble_standard_deviation.max():.6f}"
)

print()
print("Prediction/label length match:")
print(
    len(ensemble_predictions) == len(all_labels)
)

Ensemble uncertainty analysis created.

Mean probability shape:
(25684,)

Variance range:
0.000174 to 0.073641

Standard deviation range:
0.013174 to 0.271369

Prediction/label length match:
True


In [75]:
# Compare ensemble disagreement between correct and incorrect predictions.
#
# If disagreement is useful as an uncertainty signal, we would expect
# incorrect predictions to have higher average disagreement than
# correct predictions.
#
# We use standard deviation here because it is easier to interpret
# than variance: it is on the same probability scale as the model
# predictions.

correct_mask = ensemble_predictions == all_labels
incorrect_mask = ~correct_mask

correct_uncertainty = ensemble_standard_deviation[correct_mask]
incorrect_uncertainty = ensemble_standard_deviation[incorrect_mask]

print("Ensemble uncertainty by prediction outcome")
print("-------------------------------------------")

print()
print("Correct predictions:")
print("Count:", len(correct_uncertainty))
print("Mean disagreement:", f"{correct_uncertainty.mean():.6f}")
print("Median disagreement:", f"{np.median(correct_uncertainty):.6f}")

print()
print("Incorrect predictions:")
print("Count:", len(incorrect_uncertainty))
print("Mean disagreement:", f"{incorrect_uncertainty.mean():.6f}")
print("Median disagreement:", f"{np.median(incorrect_uncertainty):.6f}")

print()
print("Overall accuracy:")
print(f"{correct_mask.mean():.4f}")

Ensemble uncertainty by prediction outcome
-------------------------------------------

Correct predictions:
Count: 10880
Mean disagreement: 0.101513
Median disagreement: 0.104394

Incorrect predictions:
Count: 14804
Mean disagreement: 0.093461
Median disagreement: 0.091569

Overall accuracy:
0.4236


In [76]:
# Test whether ensemble disagreement can distinguish incorrect
# predictions from correct predictions.
#
# We treat an incorrect prediction as the "error" class.
# If disagreement is a useful uncertainty signal, higher
# disagreement should generally correspond to a higher chance
# of being incorrect.
#
# AUROC is useful here because it evaluates the ranking ability
# of the uncertainty score without choosing an arbitrary threshold.

from sklearn.metrics import roc_auc_score

# Create the target variable:
# 1 = incorrect prediction
# 0 = correct prediction
error_labels = incorrect_mask.astype(int)

# Use ensemble standard deviation as the uncertainty score.
# Higher values represent greater disagreement between models.
uncertainty_error_auroc = roc_auc_score(
    error_labels,
    ensemble_standard_deviation
)

print("Error-detection analysis")
print("------------------------")

print()
print("AUROC for disagreement detecting incorrect predictions:")
print(f"{uncertainty_error_auroc:.4f}")

print()
print("Interpretation reference:")
print("0.50 = no useful ranking signal")
print(">0.50 = higher disagreement tends to indicate errors")
print("<0.50 = higher disagreement tends to indicate correct predictions")

Error-detection analysis
------------------------

AUROC for disagreement detecting incorrect predictions:
0.4225

Interpretation reference:
0.50 = no useful ranking signal
>0.50 = higher disagreement tends to indicate errors
<0.50 = higher disagreement tends to indicate correct predictions


In [77]:
# Examine the predictions where the ensemble models disagree the most.
#
# These cases are particularly interesting for trustworthy AI because
# a model may appear confident overall while the individual models
# strongly disagree with one another.
#
# We look at the top 10% most-disagreeing cases and compare their
# error rate with the overall error rate.

disagreement_threshold = np.percentile(
    ensemble_standard_deviation,
    90
)

high_disagreement_mask = (
    ensemble_standard_deviation >= disagreement_threshold
)

high_disagreement_correct = correct_mask[high_disagreement_mask]
high_disagreement_error_rate = (
    1 - high_disagreement_correct.mean()
)

overall_error_rate = 1 - correct_mask.mean()

print("High-disagreement analysis")
print("--------------------------")

print()
print("90th-percentile disagreement threshold:")
print(f"{disagreement_threshold:.6f}")

print()
print("High-disagreement cases:")
print(high_disagreement_mask.sum())

print()
print("Error rate among high-disagreement cases:")
print(f"{high_disagreement_error_rate:.4f}")

print()
print("Overall error rate:")
print(f"{overall_error_rate:.4f}")

print()
print("Difference:")
print(
    f"{high_disagreement_error_rate - overall_error_rate:.4f}"
)

High-disagreement analysis
--------------------------

90th-percentile disagreement threshold:
0.142279

High-disagreement cases:
2569

Error rate among high-disagreement cases:
0.6236

Overall error rate:
0.5764

Difference:
0.0472


In [78]:
# Examine cases where the ensemble is confident but the individual
# models disagree strongly.
#
# These cases are especially important for trustworthy AI because
# the ensemble mean may suggest a confident prediction while the
# underlying models are not fully consistent with one another.

ensemble_confidence = np.maximum(
    ensemble_mean_probability,
    1 - ensemble_mean_probability
)

high_confidence_mask = ensemble_confidence >= 0.90

high_disagreement_confidence_mask = (
    high_confidence_mask &
    high_disagreement_mask
)

selected_count = high_disagreement_confidence_mask.sum()

if selected_count > 0:
    selected_error_rate = (
        1
        - correct_mask[high_disagreement_confidence_mask].mean()
    )

    print("High-confidence + high-disagreement analysis")
    print("----------------------------------------------")

    print()
    print("Confidence threshold:")
    print(">= 0.90")

    print()
    print("Disagreement threshold:")
    print(f">= {disagreement_threshold:.6f}")

    print()
    print("Number of cases:")
    print(selected_count)

    print()
    print("Error rate:")
    print(f"{selected_error_rate:.4f}")

else:
    print("No cases met both thresholds.")

No cases met both thresholds.


# Stage 5 Findings: Deep Ensemble Uncertainty

## Purpose

The previous stage showed that the single model could be highly confident even when its prediction was incorrect. Stage 5 therefore looked at whether model disagreement could provide an additional signal of uncertainty.

The idea was simple: instead of relying on one trained model, I trained five independently initialized copies of the same SmallCNN architecture and compared their predictions on the external RSNA dataset.

The research question was:

> Does disagreement between independently trained models provide a useful signal for identifying uncertain or incorrect predictions under external distribution shift?

---

## Experimental setup

Five models were trained using the same:

- PneumoniaMNIST training data
- Image preprocessing
- SmallCNN architecture
- Binary classification objective
- Optimizer and learning rate
- Five-epoch training procedure

The only intentional difference between the models was the random seed:

```text
42, 43, 44, 45, 46

# Trustworthy AI for Medical Imaging

## Project Summary

This project started with a simple question:

> How reliable is a medical imaging model when it is used outside the data it was originally trained on?

I wanted to look beyond accuracy and examine a few things that matter for trustworthy AI:

- How the model behaves when image appearance changes
- Whether performance transfers to an external dataset
- Whether the probabilities produced by the model are well calibrated
- Whether the model can be confidently wrong
- Whether model disagreement can provide a useful uncertainty signal

The project was carried out as a sequence of experiments, with each stage building on the previous one.

---

# Research Pipeline

The project followed this progression:

```text
Stage 1
Baseline model
      ↓
Controlled image shift
      ↓
Stage 2
External evaluation on RSNA
      ↓
Stage 3
Temperature scaling
      ↓
Stage 4
Confidence and error analysis
      ↓
Stage 5
Deep ensemble uncertainty